In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import xarray as xr

import eradiate
from eradiate.scenes.atmosphere import ParticleLayer

sns.set_style("ticks")
eradiate.set_mode("mono")

In [ ]:
xr.load_dataset("govaerts_2021-desert-aer_core_v2.nc")

In [ ]:
layer = ParticleLayer(particle_properties="govaerts_2021-desert-aer_core_v2.nc")
layer

In [ ]:
%matplotlib inline

import matplotlib.colors as mcolors
import seaborn as sns

from eradiate.scenes.atmosphere import ParticleLayer
from eradiate.units import unit_registry as ureg

sns.set_theme(style="ticks")


def plot_phase_3d(
    pp, w_nm: float, phamat_idx: int = 0, log_scale: bool = True, ax=None
):
    """
    Visualize a ParticleProperties phase function as a 3D polar surface.

    The selected phase matrix component is mapped as radial distance from
    the origin at each scattering angle.  Azimuthal symmetry is assumed, so
    the surface is a body of revolution around the z-axis.

    For components with negative values a diverging colormap (coolwarm) centred
    symmetrically at 0 is used and the radial coordinate is the magnitude.
    log_scale is ignored for such components.  For all-positive components
    (e.g. P11) the mako colormap is used; in log mode the colorbar displays
    actual values via LogNorm.

    Parameters
    ----------
    pp : ParticleProperties
    w_nm : float
      Wavelength in nm.
    phamat_idx : int
      Index into the phamat dimension selecting which phase matrix component
      to display (e.g. 0 → P11, 1 → P12, …).
    log_scale : bool
      If True (and component is all-positive), use log10 of the phase as the
      radial coordinate.  Ignored when the component has negative values.
    ax : Axes3D, optional
      Existing 3-D axes to draw into; created if None.
    """
    w = w_nm * ureg.nm
    mu, phase = pp.eval_phase(w)  # mu: (nangles,), phase: (nphamat, nangles)
    p = phase[phamat_idx]

    phamat_label = str(pp.data.phamat.values[phamat_idx])

    theta = np.arccos(np.clip(mu, -1, 1))  # scattering angle [0, π]
    phi = np.linspace(0, 2 * np.pi, 361)
    THETA, PHI = np.meshgrid(theta, phi)

    r_raw = np.tile(p, (len(phi), 1))
    has_negatives = np.any(p < 0)

    if has_negatives:
        # log_scale is not meaningful for signed components
        r = np.abs(r_raw)
        c = r_raw
        vabs = float(np.abs(c).max())
        norm = mcolors.Normalize(vmin=-vabs, vmax=vabs)
        cmap = plt.cm.coolwarm
        cbar_label = f"P{phamat_label} [1/sr]"
        log_scale = False  # suppress log annotation in title
    else:
        if log_scale:
            r = np.log10(np.maximum(r_raw, 1e-10))
            r -= r.min()
            vmin = float(np.maximum(r_raw, 1e-10).min())
            vmax = float(r_raw.max())
            norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
        else:
            r = r_raw
            norm = mcolors.Normalize(vmin=float(r_raw.min()), vmax=float(r_raw.max()))
        c = r_raw
        cmap = sns.cm.mako
        cbar_label = f"P{phamat_label} [1/sr]"

    X = r * np.sin(THETA) * np.cos(PHI)
    Y = r * np.sin(THETA) * np.sin(PHI)
    Z = r * np.cos(THETA)

    c_norm = norm(c)

    if ax is None:
        fig, ax = plt.subplots(
            1, 1, figsize=(6, 6), subplot_kw={"projection": "3d"}, layout="constrained"
        )

    ax.plot_surface(X, Y, Z, facecolors=cmap(c_norm), alpha=0.7, linewidth=0)

    mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    mappable.set_array([])
    plt.colorbar(mappable, ax=ax, shrink=0.5, pad=0.1, label=cbar_label)

    ax.set_title(
        f"Phase function P{phamat_label} − {w_nm:.1f} nm"
        + (" (log10)" if log_scale else "")
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_box_aspect([1, 1, 1])
    return ax


# --- usage ---
layer = ParticleLayer(particle_properties="govaerts_2021-desert-aer_core_v2.nc")
layer = ParticleLayer(particle_properties="REF_UB-aer_core_v2.nc")
ax = plot_phase_3d(layer.particle_properties, w_nm=550.0, log_scale=True)
plt.show()

In [ ]:
%matplotlib widget

import ipywidgets as widgets

pp = layer.particle_properties
wavelengths = pp.w.m_as("nm").tolist()
wavelength_labels = [f"{x:.1f}" for x in wavelengths]
phamat_options = [(f"P{v}", i) for i, v in enumerate(pp.data.phamat.values)]

fig = plt.figure(figsize=(6, 6), layout="constrained")


@widgets.interact(
    w_nm=widgets.SelectionSlider(
        options=list(zip(wavelength_labels, wavelengths)),
        value=wavelengths[0],
        description="λ (nm)",
        continuous_update=False,
        layout=widgets.Layout(width="350px"),
    ),
    phamat_idx=widgets.Dropdown(
        options=phamat_options,
        value=0,
        description="Component",
    ),
    log_scale=widgets.Checkbox(value=True, description="Log scale"),
)
def update(w_nm, phamat_idx, log_scale):
    fig.clear()
    ax = fig.add_subplot(111, projection="3d")
    plot_phase_3d(pp, w_nm=w_nm, phamat_idx=phamat_idx, log_scale=log_scale, ax=ax)
    fig.canvas.draw_idle()